# Baseline stream recommender

This notebook consumes `data/live_stream_features.jsonl`, normalizes the basic stream features, applies an explicit linear weight vector, and produces the top 50 live streams.

There are not yet interaction labels such as clicks or watch duration, so this is a coefficient-defined linear scoring baseline rather than a statistically fitted regression. The resulting `y_score` is the target to replace with a learned target when labels are available.

## Load exported data

Run `00_get_data.ipynb` first so that `data/live_stream_features.jsonl` exists.

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

notebook_dir = Path.cwd() if Path.cwd().name == 'notebooks' else Path.cwd() / 'notebooks'
data_path = notebook_dir / 'data' / 'live_stream_features.jsonl'
if not data_path.exists():
    raise FileNotFoundError(f'Run 00_get_data.ipynb first: {data_path}')

streams_df = pd.read_json(data_path, lines=True)
print(f'Loaded {len(streams_df)} live streams from {data_path}')
streams_df.head(3)

Loaded 1000 live streams from /Users/edwynzhou/Documents/repos/TwitchEE/notebooks/data/live_stream_features.jsonl


,id,user_id,user_login,user_name,game_id,game_name,title,language,tags,is_mature,viewer_count,started_at,stream_age_hours,viewer_log1p,tag_count,title_length,is_followed_channel,matches_recent_vod_category
0,318183054051,92038375,caedrel,Caedrel,21779,League of Legends,🔴WORLDS 2026 G2 VS HLE ELIMINATION🔴,en,"[English, LPL, LCK, LEC, Worlds, Esports, Drop...",False,287210,2026-10-30 14:55:54+00:00,9.155765,12.567972,7,35,False,False
1,319744886615,36029255,riotgames,Riot Games,21779,League of Legends,G2 vs HLE | WORLDS 2026 | Swiss Stage Day 7,en,"[Esports, English, DropsEnabled]",False,59177,2026-10-30 18:59:57+00:00,5.088265,10.988305,3,43,False,False
2,318156949730,83232866,ibai,ibai,21779,League of Legends,#WORLDS2026 | SE LA JUEGA G2 | SE LA JUEGA JOS...,es,"[Español, IRL, Chatting, KOI, DropsActivados]",False,53601,2026-10-30 19:25:30+00:00,4.662432,10.889342,5,132,False,False


## Define and normalize the baseline features

All features are normalized to `[0, 1]` before weights are applied. Viewer count uses the API export's `viewer_log1p` feature so very large channels do not dominate linearly. Stream age is inverted because newer streams are preferred.

In [6]:
feature_columns = [
    'viewer_log1p',
    'tag_count',
    'title_length',
    'stream_age_hours',
    'is_followed_channel',
    'matches_recent_vod_category',
]

missing_columns = [column for column in feature_columns if column not in streams_df.columns]
if missing_columns:
    raise ValueError(f'Export is missing baseline features: {missing_columns}')

numeric_features = streams_df[feature_columns].copy()
for column in feature_columns:
    numeric_features[column] = pd.to_numeric(numeric_features[column], errors='coerce').fillna(0)

scaler = MinMaxScaler()
normalized = pd.DataFrame(
    scaler.fit_transform(numeric_features),
    columns=feature_columns,
    index=streams_df.index,
)
normalized['stream_age_hours'] = 1 - normalized['stream_age_hours']
normalized = normalized.clip(0, 1)
normalized.add_suffix('_normalized').head(3)

,viewer_log1p_normalized,tag_count_normalized,title_length_normalized,stream_age_hours_normalized,is_followed_channel_normalized,matches_recent_vod_category_normalized
0,1.000000,0.666667,0.233577,0.806634,0.0,0.0
1,0.768113,0.222222,0.291971,0.892770,0.0,0.0
2,0.753586,0.444444,0.941606,0.901788,0.0,0.0


## Apply the linear baseline

Adjust these coefficients as the baseline evolves. Positive coefficients increase recommendation score; the coefficients sum to `1.0`, keeping `y_score` interpretable on `[0, 1]`.

In [7]:
baseline_weights = {
    'viewer_log1p': 0.50,
    'tag_count': 0.05,
    'title_length': 0.05,
    'stream_age_hours': 0.05,
    'is_followed_channel': 0.20,
    'matches_recent_vod_category': 0.15,
}

if not np.isclose(sum(baseline_weights.values()), 1.0):
    raise ValueError('Baseline weights must sum to 1.0')

weights = pd.Series(baseline_weights).reindex(feature_columns)
streams_scored_df = streams_df.copy()
streams_scored_df['y_score'] = normalized.dot(weights)
streams_scored_df = streams_scored_df.sort_values(
    ['y_score', 'viewer_count'], ascending=[False, False]
)

top_50_streams_df = streams_scored_df.head(50).reset_index(drop=True)
top_50_streams_df.insert(0, 'rank', top_50_streams_df.index + 1)
top_50_streams_df[['rank', 'user_name', 'game_name', 'title', 'viewer_count', 'y_score']].head(50)

,rank,user_name,game_name,title,viewer_count,y_score
0,1,Caedrel,League of Legends,🔴WORLDS 2026 G2 VS HLE ELIMINATION🔴,287210,0.585344
1,2,Kamet0,League of Legends,G2 ESPORTS 0 vs 1 HANWHA LIFE | WORLDS 2026 - ...,53392,0.507240
2,3,ibai,League of Legends,#WORLDS2026 | SE LA JUEGA G2 | SE LA JUEGA JOS...,53601,0.491185
3,4,TheBurntPeanut,Dead by Daylight,SPOOKY STREAM | DEAD BY DAYLIGHT | CLOAKZY X K...,31514,0.467257
4,5,Riot Games,League of Legends,G2 vs HLE | WORLDS 2026 | Swiss Stage Day 7,59177,0.454405
5,6,Tolkin,League of Legends,🟢WORLDS G2 vs HLE ELIMINATION Bo3 🟢NEUER !MERC...,30596,0.434080
6,7,HasanAbi,Just Chatting,🎃4 DAYS TILL MIDTERMS!!👻AES SURGES +7 DESPITE ...,27720,0.429099
7,8,otplol_,League of Legends,WORLDS - G2 vs HLE - MATCHS ÉLIMINATOIRES - SW...,38881,0.426724
8,9,xQc,Grand Theft Auto V,🎯LIVE🎯HERE🎯DRAMA🎯LOCK IN🎯GTA V🎯NOPIXEL V🎯MEGA ...,27731,0.421073
9,10,zackrawrr,Phantom Blade Zero,[DROPS ON] UNBANNED FOREVER PREP BIG DAY HUGE ...,26038,0.420033


In [8]:
output_path = notebook_dir / 'data' / 'baseline_top_50.jsonl'
top_50_streams_df.to_json(output_path, orient='records', lines=True, date_format='iso')
print(f'Wrote {len(top_50_streams_df)} ranked streams to {output_path}')
print('Weights:', baseline_weights)

Wrote 50 ranked streams to /Users/edwynzhou/Documents/repos/TwitchEE/notebooks/data/baseline_top_50.jsonl
Weights: {'viewer_log1p': 0.5, 'tag_count': 0.05, 'title_length': 0.05, 'stream_age_hours': 0.05, 'is_followed_channel': 0.2, 'matches_recent_vod_category': 0.15}
